# 01 — NLP Basics on Resume Text

This notebook walks through every preprocessing step used in the Resume Analyzer
pipeline. For each step you will see a **before → after** comparison so you can
understand exactly what the code is doing.

**Steps covered:**
1. Raw resume paragraph (our starting point)
2. `light_clean()` — whitespace & character normalisation
3. Tokenisation — splitting text into words
4. Sentence detection — finding sentence boundaries
5. Stop-word removal — discarding common, uninformative words
6. Lemmatisation — reducing words to their dictionary base form

> **Tip:** Run each cell with **Shift + Enter** and read the output before moving on.

In [ ]:
# ── Cell 1 : Setup ──────────────────────────────────────────────────────────
# Make sure the project root is on the Python path so we can import app modules.
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent   # notebook lives in notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.preprocess import light_clean, nlp_process

print('✅ Imports OK — project root:', PROJECT_ROOT)

In [ ]:
# ── Cell 2 : Sample Resume Paragraph ────────────────────────────────────────
# This is a realistic extract from a software-engineer resume.
# It deliberately contains messy formatting: extra spaces, a bullet, an em-dash,
# curly quotes, and a decorative separator line — all common in real PDFs.

raw_text = """
Experienced  Python developer  with 5+ years of experience  in building
scalable   back-end systems.
-------------------------------------------
• Designed and implemented RESTful APIs using FastAPI and Django\u2014reducing
  response latency by 40\u0025.
\u2018Proficient\u2019 in containerisation with Docker & Kubernetes.
Skills:  Python, Machine Learning, NLP, SQL,  TensorFlow.
"""

print('=== RAW TEXT (exactly as extracted from the resume) ===')
print(repr(raw_text))   # repr() reveals invisible chars like \n

---
## Step 1 — Light Cleaning

`light_clean()` is a *surface-level* fix.  It:
- Converts Unicode ligatures and special symbols to plain ASCII equivalents.
- Removes invisible control characters.
- Replaces fancy quotes (`\u2018`, `\u2019`), em-dashes, and bullet points.
- Strips decorative separator lines (e.g. `------`).
- Collapses multiple spaces and blank lines.

We use this output as input to **embedding models** — the sentences stay intact.

In [ ]:
# ── Cell 3 : light_clean ────────────────────────────────────────────────────
cleaned = light_clean(raw_text)

print('=== BEFORE (raw) ===')
print(raw_text)
print()
print('=== AFTER  light_clean() ===')
print(cleaned)

In [ ]:
# ── Cell 4 : What exactly changed? ──────────────────────────────────────────
# Check a few specific transformations side-by-side.

checks = [
    ('Em-dash gone',      '\u2014' not in cleaned),
    ('Curly quote gone',  '\u2018' not in cleaned and '\u2019' not in cleaned),
    ('Separator gone',    '---' not in cleaned),
    ('Bullet replaced',   '•' not in cleaned),
    ('No double spaces',  '  ' not in cleaned),
]

for label, ok in checks:
    status = '✅' if ok else '❌'
    print(f'  {status}  {label}')

---
## Step 2 — Tokenisation

**Tokenisation** = splitting a string of text into a list of individual
*tokens* (mostly words, but also punctuation marks).

spaCy is smart about edge cases:
- `FastAPI` → stays as one token (it's a proper noun).
- `back-end` → split into `back`, `-`, `end`.
- `40%` → split into `40` and `%`.

Why it matters: downstream models work on individual word units, not raw strings.

In [ ]:
# ── Cell 5 : Run nlp_process ─────────────────────────────────────────────────
# nlp_process() runs spaCy once and returns everything in a single dict.
# (This may take a few seconds on the first call while spaCy loads its model.)

result = nlp_process(cleaned)

# Show the five keys we get back
for key, value in result.items():
    print(f'{key:20s} → {value[:8]}...')  # preview first 8 items

In [ ]:
# ── Cell 6 : All tokens ─────────────────────────────────────────────────────
tokens = result['tokens']

print(f'Total tokens (excluding whitespace): {len(tokens)}')
print()
print('First 20 tokens:')
print(tokens[:20])
print()
print('Notice: punctuation (. , -) appears as its own token.')

---
## Step 3 — Sentence Detection

spaCy's **sentenciser** (dependency parser) identifies where one sentence ends
and the next begins.  This is useful when we want to embed individual sentences
rather than the whole document.

In [ ]:
# ── Cell 7 : Sentences ───────────────────────────────────────────────────────
sentences = result['sentences']

print(f'Number of sentences detected: {len(sentences)}')
print()
for i, sent in enumerate(sentences, 1):
    print(f'  Sentence {i}: {sent[:90]}...' if len(sent) > 90 else f'  Sentence {i}: {sent}')

---
## Step 4 — Stop-Word Removal

**Stop words** are common words that carry little meaning by themselves:
*the, a, is, to, with, and, of, in, …*

Removing them:
- Reduces noise in keyword extraction and TF-IDF models.
- Keeps only the content-bearing words that describe skills and experience.

In [ ]:
# ── Cell 8 : Stop-word removal ───────────────────────────────────────────────
filtered_tokens = result['filtered_tokens']

print('=== ALL TOKENS (with stop words) ===')
print(result['tokens'][:25])
print()
print('=== AFTER stop-word + punctuation removal ===')
print(filtered_tokens[:25])
print()

removed_count = len(result['tokens']) - len(filtered_tokens)
print(f'Tokens removed: {removed_count}  '
      f'({removed_count / len(result["tokens"]):.0%} of total)')

In [ ]:
# ── Cell 9 : Which specific words were dropped? ──────────────────────────────
import spacy
nlp = spacy.load('en_core_web_sm')

all_words = [tok.text for tok in nlp(cleaned) if not tok.is_space and not tok.is_punct]
kept_set  = set(result['filtered_tokens'])

dropped = [w for w in all_words if w not in kept_set]
print('Words dropped as stop words or numbers:')
print(sorted(set(dropped)))

---
## Step 5 — Lemmatisation

**Lemmatisation** reduces each word to its *base dictionary form* (its *lemma*):

| Surface Form | Lemma |
|---|---|
| running | run |
| implemented | implement |
| designing | design |
| reduces | reduce |
| systems | system |

Why it matters: `"builds"`, `"built"`, and `"building"` all reduce to `"build"`,
so they match as the same skill keyword.

In [ ]:
# ── Cell 10 : Lemmatisation comparison ──────────────────────────────────────
print('=== BEFORE lemmatisation (filtered tokens) ===')
print(result['filtered_tokens'])
print()
print('=== AFTER  lemmatisation (filtered lemmas, lowercased) ===')
print(result['filtered_lemmas'])

In [ ]:
# ── Cell 11 : Side-by-side token → lemma table ──────────────────────────────
changed = [
    (tok, lem)
    for tok, lem in zip(result['filtered_tokens'], result['filtered_lemmas'])
    if tok.lower() != lem
]

print(f'Words whose form changed after lemmatisation: {len(changed)}')
print()
print(f'{"Surface form":<22}  →  {"Lemma"}')
print('-' * 40)
for tok, lem in changed:
    print(f'{tok:<22}  →  {lem}')

---
## Step 6 — Full Pipeline Summary

Here is every step in one place so you can see the complete journey
from raw document text to analysis-ready lemmas.

In [ ]:
# ── Cell 12 : Full pipeline end-to-end ──────────────────────────────────────
SEP = '\n' + '─' * 60 + '\n'

print('① RAW TEXT')
print(raw_text.strip())

print(SEP + '② AFTER light_clean()  [ready for embeddings]')
print(cleaned)

print(SEP + '③ TOKENS  (all words, punctuation excluded)')
print(result['tokens'])

print(SEP + '④ SENTENCES')
for s in result['sentences']:
    print('  »', s)

print(SEP + '⑤ FILTERED TOKENS  (stop words removed)')
print(result['filtered_tokens'])

print(SEP + '⑥ FILTERED LEMMAS  [ready for keyword / skill extraction]')
print(result['filtered_lemmas'])

In [ ]:
# ── Cell 13 : Try it on your own text ───────────────────────────────────────
# Paste any text from your own resume below and re-run this cell.

my_text = """
Developed and deployed a real-time fraud detection system using Python and
scikit-learn, achieving 95% precision and reducing false positives by 30%.
Managed a team of 4 engineers and collaborated with product stakeholders.
"""

cleaned_mine  = light_clean(my_text)
result_mine   = nlp_process(cleaned_mine)

print('Filtered lemmas (keywords):')
print(result_mine['filtered_lemmas'])